In [1]:
!pip install -q transformers peft datasets scipy soundfile accelerate tqdm librosa

## Pre-processing step:
- Loading audio clips and resampling to 32kHz range
- Chunking into 30 second audio clips
- Creating metadata.csv file

In [4]:
import os
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm

# 1. Creating directory structure
DATA_DIR = "./hindustani_data"
CHUNKS_DIR = os.path.join(DATA_DIR, "chunks")
os.makedirs(CHUNKS_DIR, exist_ok=True)

# 2. Dictionary mapping each file to its label
audio_labels = {
    "audio1.mp3": "Traditional Hindustani classical sitar performance, rhythmic instrumental creating a harmonious and peaceful tune and melody.",
    "audio2.mp3": "Fast tempo traditional Hindustani classical raga-styled ensemble performance, with tabla, veena and sitar, rhythmic and instrumental, both string and percussion instruments",
    "audio3.mp3": "Modern Hindustani solo tabla performance, isolated and upbeat tabla instrumental",
    "audio4.mp3": "Traditional Hindustani classical instrumental performance, complete with sitar and tabla, traditional raga structure.",
    "audio5.mp3": "Classical Hindustani instrumental ensemble, purely instrumental with string and percussion, tabla and sitar performance with raga-structure."
}

target_sr = 32000  # MusicGen's native sampling rate
chunk_duration = 30  # seconds
metadata_records = []
chunk_idx = 0

print(f"Processing {len(audio_labels)} local MP3 files into 30-second chunks...")

for file_name, specific_label in audio_labels.items():
    if not os.path.exists(file_name):
        print(f"Error: {file_name} not found in current directory. Please upload it first.")
        continue

    print(f"Processing: {file_name} -> Label: '{specific_label}'")
    # Load audio and automatically resample to 32kHz
    y, sr = librosa.load(file_name, sr=target_sr)
    samples_per_chunk = chunk_duration * target_sr

    # Iterate through audio in 30-second steps
    for start_sample in range(0, len(y), samples_per_chunk):
        end_sample = start_sample + samples_per_chunk

        # Drop trailing fragments shorter than 30 seconds to maintain data uniformity
        if end_sample > len(y):
            break

        chunk = y[start_sample:end_sample]
        relative_chunk_path = f"chunks/chunk_{chunk_idx}.wav"
        full_chunk_path = os.path.join(DATA_DIR, relative_chunk_path)

        # Save chunk as high-quality WAV
        sf.write(full_chunk_path, chunk, target_sr)

        # Apply the specific label for this file to all its chunks
        metadata_records.append({
            "file_name": relative_chunk_path,
            "text": specific_label
        })
        chunk_idx += 1

# 3. Write metadata file
metadata_df = pd.DataFrame(metadata_records)
metadata_df.to_csv(os.path.join(DATA_DIR, "metadata.csv"), index=False)

print(f"Dataset preparation complete! Generated {chunk_idx} chunks and saved metadata.csv.")

Processing 5 local MP3 files into 30-second chunks...
Processing: audio1.mp3 -> Label: 'Traditional Hindustani classical sitar performance, rhythmic instrumental creating a harmonious and peaceful tune and melody.'
Processing: audio2.mp3 -> Label: 'Fast tempo traditional Hindustani classical raga-styled ensemble performance, with tabla, veena and sitar, rhythmic and instrumental, both string and percussion instruments'
Processing: audio3.mp3 -> Label: 'Modern Hindustani solo tabla performance, isolated and upbeat tabla instrumental'
Processing: audio4.mp3 -> Label: 'Traditional Hindustani classical instrumental performance, complete with sitar and tabla, traditional raga structure.'
Processing: audio5.mp3 -> Label: 'Classical Hindustani instrumental ensemble, purely instrumental with string and percussion, tabla and sitar performance with raga-structure.'
Dataset preparation complete! Generated 71 chunks and saved metadata.csv.


In [5]:
!zip -r /content/chunks_and_metadata.zip /content/hindustani_data

  adding: content/hindustani_data/ (stored 0%)
  adding: content/hindustani_data/chunks/ (stored 0%)
  adding: content/hindustani_data/chunks/chunk_46.wav (deflated 4%)
  adding: content/hindustani_data/chunks/chunk_9.wav (deflated 18%)
  adding: content/hindustani_data/chunks/chunk_28.wav (deflated 9%)
  adding: content/hindustani_data/chunks/chunk_45.wav (deflated 2%)
  adding: content/hindustani_data/chunks/chunk_0.wav (deflated 20%)
  adding: content/hindustani_data/chunks/chunk_26.wav (deflated 6%)
  adding: content/hindustani_data/chunks/chunk_67.wav (deflated 10%)
  adding: content/hindustani_data/chunks/chunk_52.wav (deflated 18%)
  adding: content/hindustani_data/chunks/chunk_39.wav (deflated 2%)
  adding: content/hindustani_data/chunks/chunk_11.wav (deflated 17%)
  adding: content/hindustani_data/chunks/chunk_8.wav (deflated 20%)
  adding: content/hindustani_data/chunks/chunk_69.wav (deflated 10%)
  adding: content/hindustani_data/chunks/chunk_60.wav (deflated 8%)
  adding: c

## Loading base model and generating baseline outputs:
- **Prompt1**: "tabla_solo": "A highly detailed solo Indian classical Hindustani modern tabla instrumental, featuring intricate rhythmic patterns, fast-paced beats, and clear, resonant percussion."
- **Prompt2**: "sitar_duet": "A traditional Indian Hindustani classical raga-styled instrumental, featuring a slow, expressive sitar melody beautifully accompanied by steady, rhythmic tabla beats."
- **Prompt3**: "grand_ensemble": "A grand Indian traditional raga-styled ensemble performance, featuring a rich tapestry of veena, sitar, tabla, and other diverse string and percussion instruments, creating a vibrant rhythm and majestic melody."

In [6]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("./baseline_outputs", exist_ok=True)

print("Loading base model for baseline generations...")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

# Defining the prompts
prompts = {
    "tabla_solo": "A highly detailed solo Indian classical Hindustani modern tabla instrumental, featuring intricate rhythmic patterns, fast-paced beats, and clear, resonant percussion.",

    "sitar_duet": "A traditional Indian Hindustani classical raga-styled instrumental, featuring a slow, expressive sitar melody beautifully accompanied by steady, rhythmic tabla beats.",

    "grand_ensemble": "A grand Indian traditional raga-styled ensemble performance, featuring a rich tapestry of veena, sitar, tabla, and other diverse string and percussion instruments, creating a vibrant rhythm and majestic melody."
}

print("Generating baseline audio samples...")
model.eval()
with torch.no_grad():
    for label, prompt_text in prompts.items():
        print(f"Generating for prompt: '{label}'...")
        inputs = processor(text=[prompt_text], padding=True, return_tensors="pt").to(device)

        # Generating 10 seconds of audio (500 tokens)
        audio_values = model.generate(**inputs, max_new_tokens=500, guidance_scale=3.0, do_sample=True, temperature=1.0)

        output_path = f"./baseline_outputs/baseline_{label}.wav"
        scipy.io.wavfile.write(output_path, rate=32000, data=audio_values[0, 0].cpu().numpy())
        print(f"Saved: {output_path}")

print("All baseline samples generated successfully.")

Loading base model for baseline generations...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Generating baseline audio samples...
Generating for prompt: 'tabla_solo'...
Saved: ./baseline_outputs/baseline_tabla_solo.wav
Generating for prompt: 'sitar_duet'...
Saved: ./baseline_outputs/baseline_sitar_duet.wav
Generating for prompt: 'grand_ensemble'...
Saved: ./baseline_outputs/baseline_grand_ensemble.wav
All baseline samples generated successfully.


## Fine-tuning through LoRA and custom Hyperparameters

* **Rank (`r`): 16**
  
  Dimensional complexity of the low-rank matrices.
* **Alpha (`lora_alpha`): 16**
  
  Scaled to a 1:1 ratio with rank to stabilize weights and eliminate noise.
* **Target Modules: `["q_proj", "v_proj"]`**
  
  Applies adaptation strictly to the decoder's attention layer projections.
* **Dropout: 0.05**
  
  Randomly deactivates 5% of adapter units to prevent overfitting.
* **Learning Rate: 2e-5**
  
  Conservative step size to preserve pre-trained acoustic structures.
* **Epochs: 3**
  
  Capped low to stop the model from memorizing the limited audio files.
* **Batch Strategy: 1 (Accumulation Steps: 4)**
  
  Simulates a stable batch size of 4 to stay within T4 GPU memory limits.

In [7]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.2 MB/s eta 0:00:00


In [17]:
import torch
import gc
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, Audio
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# Force clear any lingering memory in the GPU before starting
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Initializing Core Components
print("Initializing model architecture...")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

# --- THE DEEP CONFIGURATION FIX ---
start_token = model.config.decoder.bos_token_id or model.config.decoder.pad_token_id
model.config.decoder.decoder_start_token_id = start_token
model.config.decoder_start_token_id = start_token
model.config.pad_token_id = model.config.decoder.pad_token_id
# ----------------------------------

# 2. Configuring Low-Rank Adaptation (LoRA)
# UPDATED: lora_alpha changed from 32 to 16 for a safer 1:1 weight scaling ratio
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. Ingest Local Dataset Structure
print("Loading local chunk data...")
dataset = load_dataset("audiofolder", data_dir="./hindustani_data", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=32000))

def collate_fn(batch):
    texts = [item["text"] for item in batch]
    audios = [item["audio"]["array"] for item in batch]
    inputs = processor(text=texts, audio=audios, sampling_rate=32000, padding=True, return_tensors="pt")
    return inputs

dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

# UPDATED: Learning rate dropped from 1e-4 to 2e-5 to prevent weight explosion
optimizer = AdamW(model.parameters(), lr=2e-5)

# UPDATED: Reduced from 5 to 3 to prevent overfitting on small datasets
epochs = 3
accumulation_steps = 4

# 4. Training Execution
print("Beginning parameter optimization...")
model.train()

for epoch in range(epochs):
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    optimizer.zero_grad()

    for i, batch in enumerate(loop):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask").to(device) if "attention_mask" in batch else None
        input_values = batch["input_values"].to(device)

        with torch.no_grad():
            audio_codes = model.audio_encoder.encode(input_values, return_dict=True).audio_codes
            labels = audio_codes.squeeze(1)
            labels = labels[:, :, :1500]

            # --- THE DIMENSION FIX ---
            labels = labels.transpose(1, 2)
            # -------------------------

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        loss = outputs.loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()

        loop.set_postfix(loss=loss.item() * accumulation_steps)

# 5. Save Learned Weights
print("Saving fine-tuned adapters...")
model.save_pretrained("./hindustani-lora-local")
print("Optimization routine completed.")

Initializing model architecture...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 3,145,728 || all params: 590,030,402 || trainable%: 0.5331
Loading local chunk data...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Beginning parameter optimization...


Epoch 3/3: 100%|██████████| 71/71 [01:29<00:00,  1.26s/it, loss=8.67]


Saving fine-tuned adapters...
Optimization routine completed.


## Using local LoRA model to generate final outputs post fine-tuning with the same prompts:
- **Prompt1**: "tabla_solo": "A highly detailed solo Indian classical Hindustani modern tabla instrumental, featuring intricate rhythmic patterns, fast-paced beats, and clear, resonant percussion."
- **Prompt2**: "sitar_duet": "A traditional Indian Hindustani classical raga-styled instrumental, featuring a slow, expressive sitar melody beautifully accompanied by steady, rhythmic tabla beats."
- **Prompt3**: "grand_ensemble": "A grand Indian traditional raga-styled ensemble performance, featuring a rich tapestry of veena, sitar, tabla, and other diverse string and percussion instruments, creating a vibrant rhythm and majestic melody."

In [19]:
import torch
import gc
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import PeftModel
import scipy.io.wavfile
import os

# 1. Clearing memory before generation
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("./finetuned_outputs", exist_ok=True)

print("Loading base model...")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
base_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

print("Applying your custom Hindustani LoRA adapter...")
# UPDATED: Loading the correct Hindustani adapter weights
model = PeftModel.from_pretrained(base_model, "./hindustani-lora-local").to(device)

# 2. Define the exact match evaluation prompts (Character-matched to Cell 3)
prompts = {
    "tabla_solo": "A highly detailed solo Indian classical Hindustani modern tabla instrumental, featuring intricate rhythmic patterns, fast-paced beats, and clear, resonant percussion.",

    "sitar_duet": "A traditional Indian Hindustani classical raga-styled instrumental, featuring a slow, expressive sitar melody beautifully accompanied by steady, rhythmic tabla beats.",

    "grand_ensemble": "A grand Indian traditional raga-styled ensemble performance, featuring a rich tapestry of veena, sitar, tabla, and other diverse string and percussion instruments, creating a vibrant rhythm and majestic melody."
}

# 3. Generating Evaluation Samples
print("Generating fine-tuned audio samples...")
model.eval()
with torch.no_grad():
    for label, prompt_text in prompts.items():
        print(f"Generating for prompt: '{label}'...")
        inputs = processor(text=[prompt_text], padding=True, return_tensors="pt").to(device)

        # Generating 500 tokens yields about 10 seconds of audio
        audio_values = model.generate(**inputs, max_new_tokens=500, guidance_scale=3, do_sample=True, temperature=1)

        output_path = f"./finetuned_outputs/finetuned_{label}.wav"
        scipy.io.wavfile.write(output_path, rate=32000, data=audio_values[0, 0].cpu().numpy())
        print(f"Saved: {output_path}")

print("\nEvaluation pipeline complete! Ready for comparison.")

Loading base model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Applying your custom Hindustani LoRA adapter...
Generating fine-tuned audio samples...
Generating for prompt: 'tabla_solo'...
Saved: ./finetuned_outputs/finetuned_tabla_solo.wav
Generating for prompt: 'sitar_duet'...
Saved: ./finetuned_outputs/finetuned_sitar_duet.wav
Generating for prompt: 'grand_ensemble'...
Saved: ./finetuned_outputs/finetuned_grand_ensemble.wav

Evaluation pipeline complete! Ready for comparison.
